# الدرس الثامن: الإخراج المهيكل باستخدام فئات البيانات القياسية (Python Dataclasses)

## المقدمة والاهداف التعليمية
في هذا الدفتر، سنتعلم كيفية استخدام ميزة الإخراج المهيكل مع كلاسات البيانات الاصلية في بايثون (`dataclasses`).

## لماذا تدعم LangChain استخدام `@dataclass`؟
1. فئات البيانات (`@dataclass`) مدمجة في مكتبة بايثون القياسية منذ الاصدار 3.7 دون الحاجة لاي حزم اضافية.
2. تمنحك مزايا الفئات البرمجية (الوصول للحقول عبر النقطة مثل `obj.attribute`) مع الحفاظ على كود نظيف وبسيط.
3. التوافق العالي مع قواعد البيانات القديمة ومكتبات الـ ORM مثل SQLAlchemy و dataclass-based architectures.

## الخطوة 1: تهيئة البيئة واستيراد dataclass
نقوم باستيراد ديكوريتور `dataclass` ومكتبات التوصيف.

In [ ]:
import os
from dataclasses import dataclass, asdict
from typing import List
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq", temperature=0)

## الخطوة 2: تعريف فئة البيانات عبر `@dataclass`
نعرف هيكلا لاستخراج بيانات رحلات السفر الجوية.

In [ ]:
@dataclass
class FlightBookingDetails:
    """Structured extraction of flight itinerary information."""
    departure_airport: str
    destination_airport: str
    airline: str
    flight_number: str
    cabin_class: str
    total_stops: int

print("Dataclass schema defined.")

## الخطوة 3: تمرير فئة البيانات الى `with_structured_output`
تقوم LangChain باستنباط مخطط JSON من الحقول وتغذية النموذج به ثم اعادة كائن من نفس فئة الـ Dataclass.

In [ ]:
dataclass_llm = model.with_structured_output(FlightBookingDetails)

itinerary_text = """
Confirmation Notice: You are booked on British Airways flight BA2490 flying from London Heathrow (LHR)
directly to Dubai International (DXB). You will be seated in Business Class with zero layovers.
"""

flight_info = dataclass_llm.invoke(itinerary_text)

print("Parsed Instance:", flight_info)
print("Type:", type(flight_info))
print(f"Route: {flight_info.departure_airport} -> {flight_info.destination_airport}")
print(f"Airline & Flight: {flight_info.airline} ({flight_info.flight_number})")
print(f"Class: {flight_info.cabin_class}, Stops: {flight_info.total_stops}")

## الخطوة 4: التحويل الى قاموس باستخدام `asdict`
توفر مكتبة `dataclasses` الدالة الجاهزة `asdict` لتحويل الكائن فورا الى قاموس قابل للارسال عبر الشبكة او التخزين.

In [ ]:
flight_dict = asdict(flight_info)
print("Dictionary representation:")
print(flight_dict)